Part 1: Setup

In [121]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Imports + Stopwords

In [122]:
!pip install gensim

In [123]:
!pip install faiss-cpu

In [125]:
# Standard library
import os
import re
import ast
from collections import Counter

# Scientific stack
import numpy as np
import pandas as pd

# PyTorch
import torch

# NLP utilities (NLTK)
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords")
stop_words = set(stopwords.words("english"))

# Gensim
from gensim.models import Word2Vec

# Classical ML (scikit-learn)
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support
)

# FAISS (retrieval)
import faiss

# Hugging Face Datasets
from datasets import Dataset as HFDataset

# Transformers (classification + generation)
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed
)

# Sentence Transformers (RAG embeddings)
from sentence_transformers import SentenceTransformer

# Environment & reproducibility
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

SEED = 42
set_seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Data Directories (adjust if needed)

In [126]:
BASE = "/content/drive/MyDrive/BSP_S5/"

# Dataset paths
train_path = f"{BASE}/dataset/data/train.tsv"
dev_path   = f"{BASE}/dataset/data/dev.tsv"
test_path  = f"{BASE}/dataset/data/test.tsv"

Labels

In [127]:
possible_label_columns = ["admiration", "amusement", "anger", "annoyance", "approval", "caring", "confusion", "curiosity", "desire",
                          "disappointment", "disapproval", "disgust", "embarassment", "excitement", "fear", "gratitude", "grief",
                          "joy", "love", "nervousness", "optimism", "pride", "realization", "relief", "remorse", "sadness", "surprise",
                          "neutral"]

text_column_candidates = ["text", "id", "author", "subreddit", "link_id", "parent_id", "created_utc", "rater_id", "example_very_unclear",]

LABELS = list(possible_label_columns)

NUM_LABELS = len(LABELS)

# Sanity check
print(f"Number of labels: {NUM_LABELS}")
print("First 10 labels:", LABELS[:10])

Number of labels: 28
First 10 labels: ['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment']


NLTK Setup

In [128]:
try:
    _ = stopwords.words('english')
except LookupError:
    nltk.download('stopwords')

Data Preprocessing (Cleaning the Dataset Data)

In [129]:
def preprocess_data(text):
    """Turn a raw text string into a list of tokens."""

    if text is None:
        return []

    # Ensure string + lowercase
    text = str(text).lower()

    # Remove URLs and emails
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)

    # Keep only letters and spaces
    text = re.sub(r"[^a-z\s]", " ", text)

    # Collapse multiple spaces
    text = re.sub(r"\s+", " ", text).strip()

    # Split into tokens
    tokens = text.split()

    # Remove stopwords
    tokens = [t for t in tokens if t not in stop_words]

    return tokens


Choosing Columns

In [130]:
# Identify which columns contain text and label columns (multi-label compatible)
def choose_columns(df: pd.DataFrame):
    # Picking the text column
    text_col = None
    for c in text_column_candidates:
        if c in df.columns:
            text_col = c
            break
    if text_col is None:
        raise ValueError(f"Could not find a text column in {df.columns.tolist()}")

    # Pick all label columns that exist in this dataset
    label_cols = [c for c in possible_label_columns if c in df.columns]

    if not label_cols:
        raise ValueError(f"Could not find any label columns in {df.columns.tolist()}")

    return text_col, label_cols

Parsing (GoEmotions is a multi-labeled dataset)

In [131]:
def parse_label_ids(cell):
    if pd.isna(cell):
        return []
    s = str(cell).strip()
    # Handle list-like strings
    if s.startswith('[') and s.endswith(']'):
        try:
            vals = ast.literal_eval(s)
            return [int(v) for v in vals]
        except Exception:
            pass
    # Fallback: comma-separated
    return [int(x) for x in s.replace(' ', '').split(',') if x != '']

Filtering out rows without any label

In [132]:
def make_single_label(df, label_col):
    labels_list = df[label_col].apply(parse_label_ids)
    mask_has_label = labels_list.apply(lambda xs: len(xs) > 0)
    df = df[mask_has_label].copy()
    df["primary_label"] = labels_list[mask_has_label].apply(lambda xs: xs[0])
    return df

def load_split(path):
    # The tsv files don't have header, therefore we name the columns explicitly
    df = pd.read_csv(path, sep="\t", header=None,
                     names=["text", "label_ids", "row_id"],
                     engine="python")

    # Parse label_ids into list[int]
    ids_list = df["label_ids"].apply(parse_label_ids)

    # Keep rows that have at least one label
    mask = ids_list.apply(lambda xs: len(xs) > 0)
    df = df[mask].copy()

    # For single-label baseline: take the first id
    df["primary_label"] = ids_list[mask].apply(lambda xs: xs[0])

    text_col = "text"
    y_col = "primary_label"
    return df, text_col, y_col

Vectorizer (Converts raw text into numeric values)

In [133]:
def vectorizer():
    return TfidfVectorizer(
        analyzer=preprocess_data,
        ngram_range=(1, 2),
        min_df=1,
        max_df=0.95
    )

Building Models KNN, SVM, Naive Bayes, Random Forest and Multinomial Logistic Regression

In [134]:
# Defining 4 baselines
def build_models():
    return {
        # When predicting, KNN looks for the 5 nearest samples (neighbors) in the training data to the new text vector.
        # Measures distance (in TF-IDF feature space)
        "KNN": KNeighborsClassifier(
            n_neighbors=5,
            weights="distance", # closer neighbors get more influence in the vote
            n_jobs=-1,        # use all CPU cores
        ),

        # Support Vector Machine. strong test baseline
        # Learns a hyperplane (decision boundary for separating data points into different classes) that separates classes with the maximum margin
        # Finds a weight vector "w" and bias "b" such that:
        # w · x + b > 0 for one class and
        # w · x + b < 0 for another
        "SVM": LinearSVC(),

         # Logistic Regression on top of TF-IDF + SVD (reduced dense features)
        "LogReg_SVD": LogisticRegression(
            max_iter=2000,
            n_jobs=-1,
            solver="lbfgs",
            multi_class="auto",
        ),
        # Logistic Regression with word2vec
        "LogReg_W2V": LogisticRegression(
            max_iter=2000,
            n_jobs=-1,
            solver="lbfgs",
            multi_class="auto",
        ),

        # Naive Bayes. fast BoW baseline
        # Assumes the words (features) in a document are conditionally independent given the class
        # With TF-IDF, it approximates how likely each class is given the observed word frequencies.
        "NaiveBayes": MultinomialNB(),

        # Random Forest.
        # Each tree predicts a class. The final prediction is the majority vote across trees
        "RandomForest": RandomForestClassifier(
            n_estimators=100,      # 100 trees is enough as baseline
            max_depth=25,          # limit depth to avoid huge trees
            max_features="sqrt",   # standard RF setting
            n_jobs=-1,
            random_state=42,
        ),
    }

RoBERTa Dataset Helper

Handling Training and Evaluation of one Model (returns scores)

In [135]:
def evaluate_model(clf, X_train, y_train, X_eval, y_eval):
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_eval)
    acc = accuracy_score(y_eval, y_pred)
    pr, rc, f1, _ = precision_recall_fscore_support(y_eval, y_pred, average="macro", zero_division=0)

    return acc, pr, rc, f1 # accuracy, precision, recall and F1

Part 2: Program Procedure (Defining "main()")

Loading and Preparing Data

In [136]:
train_df, text_col, y_col = load_split(train_path)
dev_df, _t, _y = load_split(dev_path)
test_df, _t2, _y2 = load_split(test_path)

for df in (train_df, dev_df, test_df):
    df[text_col] = df[text_col].fillna("").astype(str)
    df[text_col] = df[text_col].str.strip()

Multi-label targets for RoBERTa

In [137]:
NUM_LABELS = 28  # GoEmotions

def to_multihot(label_indices, num_labels=NUM_LABELS):
    y = np.zeros(num_labels, dtype=np.int64)
    for idx in label_indices:
        if 0 <= idx < num_labels:
            y[idx] = 1
    return y

train_label_lists = train_df["label_ids"].apply(parse_label_ids)
dev_label_lists   = dev_df["label_ids"].apply(parse_label_ids)
test_label_lists  = test_df["label_ids"].apply(parse_label_ids)

y_train_ml = np.stack([to_multihot(xs) for xs in train_label_lists])
y_dev_ml   = np.stack([to_multihot(xs) for xs in dev_label_lists])
y_test_ml  = np.stack([to_multihot(xs) for xs in test_label_lists])

y_train_ml = y_train_ml.astype(np.float32)
y_dev_ml   = y_dev_ml.astype(np.float32)
y_test_ml  = y_test_ml.astype(np.float32)

# y_train_ml is shape (N, 28) and float32
pos_counts = y_train_ml.sum(axis=0)
neg_counts = y_train_ml.shape[0] - pos_counts

# avoid division by zero
pos_weight_np = neg_counts / np.clip(pos_counts, 1.0, None)

Label encoding

In [138]:
le = LabelEncoder()
le.fit(pd.concat([train_df[y_col], dev_df[y_col], test_df[y_col]], axis=0))

X_train_text = train_df[text_col].tolist()
y_train = le.transform(train_df[y_col])

X_dev_text   = dev_df[text_col].tolist()
y_dev = le.transform(dev_df[y_col])

X_test_text  = test_df[text_col].tolist()
y_test = le.transform(test_df[y_col])

print("Train docs:", len(X_train_text))
print("Example train text:", X_train_text[0][:200])

Train docs: 43410
Example train text: My favourite food is anything I didn't have to cook myself.


Text Vectorization (with TF-IDF)

In [139]:
tfidf = vectorizer()

X_train = tfidf.fit_transform(X_train_text)   # learn vocab + transform train
X_dev   = tfidf.transform(X_dev_text)        # transform dev
X_test  = tfidf.transform(X_test_text)       # transform test

# use all training samples for both KNN and Random Forest
k_train_max = X_train.shape[0]
rf_train_max = X_train.shape[0]

X_train_knn = X_train[:k_train_max]
y_train_knn = y_train[:k_train_max]

X_train_rf = X_train[:rf_train_max]
y_train_rf = y_train[:rf_train_max]

svd = TruncatedSVD(n_components=300, random_state=42)
X_train_svd = svd.fit_transform(X_train)
X_dev_svd = svd.transform(X_dev)
X_test_svd = svd.transform(X_test)

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:533: UserWarning: The parameter 'ngram_range' will not be used since 'analyzer' is callable'
  warnings.warn(


word2vec: Text Tokenization

In [140]:
train_tokens = [text.split() for text in X_train_text]
dev_tokens   = [text.split() for text in X_dev_text]
test_tokens  = [text.split() for text in X_test_text]

Training word2vec on a training dataset

In [141]:
w2v_size = 300

w2v_model = Word2Vec(
    sentences=train_tokens,
    vector_size=w2v_size,
    window=5,
    min_count=2,
    workers=4,
    sg=1,          # 1 = skip-gram; 0 = CBOW
    epochs=10,
    seed=42
)

Helper to convert a list of tokens into a one-sentence vector

In [142]:
def sentence_vector(tokens, model, vector_size):
    # Average word vectors for tokens; return zeros if no token is in vocab
    vectors = [model.wv[w] for w in tokens if w in model.wv]
    if not vectors:
        return np.zeros(vector_size, dtype="float32")
    return np.mean(vectors, axis=0)

Building sentence embeddings with the contribution of word2vec

In [143]:
X_train_w2v = np.vstack([
    sentence_vector(tokens, w2v_model, w2v_size) for tokens in train_tokens
])

X_dev_w2v = np.vstack([
    sentence_vector(tokens, w2v_model, w2v_size) for tokens in dev_tokens
])

X_test_w2v = np.vstack([
    sentence_vector(tokens, w2v_model, w2v_size) for tokens in test_tokens
])

print("W2V train, dev and test shapes:",
      X_train_w2v.shape, X_dev_w2v.shape, X_test_w2v.shape)

W2V train, dev and test shapes: (43410, 300) (5426, 300) (5427, 300)


# Defining Models and Containers

# Hyperparameter tuning (SVM, LogReg_SVD, LogReg_W2V)

Tuning Configuration + Parameters grid

In [144]:
# Hyperparameter tuning flag
do_tuning = True   # set to False to skip tuning

# Hyperparameters to test for each model
param_grids = {
    "SVM": {"C": [0.1, 1.0, 10.0]},
    "LogReg_SVD": {"C": [0.1, 1.0, 10.0]},
    "LogReg_W2V": {"C": [0.1, 1.0, 10.0]},
}

Perform Tuning (3-fold CV)

In [145]:
if do_tuning:
    print(" Hyperparameter tuning (3-fold CV, F1-macro) ")

    models = build_models()

    tuned_models = {}

    for name, model in models.items():

        # If model not in tuning grid, keep as-is
        if name not in param_grids:
            tuned_models[name] = model
            continue

        # Select correct feature set depending on model
        if name == "SVM":
            Xtr, ytr = X_train, y_train              # TF-IDF
        elif name == "LogReg_SVD":
            Xtr, ytr = X_train_svd, y_train          # TF-IDF + SVD
        elif name == "LogReg_W2V":
            Xtr, ytr = X_train_w2v, y_train          # Word2Vec features
        else:
            Xtr, ytr = X_train, y_train              # fallback

        print(f"Tuning {name}...")

        grid = GridSearchCV(
            estimator=model,
            param_grid=param_grids[name],
            scoring="f1_macro",
            cv=3,
            n_jobs=-1,
            verbose=1,
        )

        grid.fit(Xtr, ytr)

        print(f"{name} best params: {grid.best_params_}")
        print(f"{name} best CV F1: {grid.best_score_:.4f}\n")

        tuned_models[name] = grid.best_estimator_

    # Replace original models with tuned ones
    models = tuned_models

    print("  Tuning completed  ")

 Hyperparameter tuning (3-fold CV, F1-macro) 
Tuning SVM...
Fitting 3 folds for each of 3 candidates, totalling 9 fits
SVM best params: {'C': 1.0}
SVM best CV F1: 0.3314

Tuning LogReg_SVD...
Fitting 3 folds for each of 3 candidates, totalling 9 fits


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


LogReg_SVD best params: {'C': 10.0}
LogReg_SVD best CV F1: 0.2732

Tuning LogReg_W2V...
Fitting 3 folds for each of 3 candidates, totalling 9 fits


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


LogReg_W2V best params: {'C': 10.0}
LogReg_W2V best CV F1: 0.1978

  Tuning completed  


Prepare Result Containers

In [146]:
results_dev = []
results_test = []

# Running all models (KNN, SVM, Naive Bayes and Random Forest) on the Tuning phase

In [147]:
# Tuning Phase
print("   Evaluation on the DEV dataset   ")
for name, model in models.items():
    if name == "KNN":
        Xtr, ytr, Xd = X_train_knn, y_train_knn, X_dev
    elif name == "RandomForest":
        Xtr, ytr, Xd = X_train_rf, y_train_rf, X_dev
    elif name == "LogReg_SVD":
        Xtr, ytr, Xd = X_train_svd, y_train, X_dev_svd
    elif name == "LogReg_W2V":
        Xtr, ytr, Xd = X_train_w2v, y_train, X_dev_w2v
    else:
        Xtr, ytr, Xd = X_train, y_train, X_dev

    acc, pr, rc, f1 = evaluate_model(model, Xtr, ytr, Xd, y_dev)
    print(f"{name:12s}  Acc={acc:.4f}  Prec={pr:.4f}  Rec={rc:.4f}  F1={f1:.4f}")

    # store for table
    results_dev.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": pr,
        "Recall": rc,
        "F1": f1,
    })


   Evaluation on the DEV dataset   
KNN           Acc=0.3111  Prec=0.3257  Rec=0.1146  F1=0.1451
SVM           Acc=0.4884  Prec=0.4202  Rec=0.3194  F1=0.3451


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


LogReg_SVD    Acc=0.4921  Prec=0.4350  Rec=0.2693  F1=0.3000


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


LogReg_W2V    Acc=0.4062  Prec=0.3283  Rec=0.1793  F1=0.2077
NaiveBayes    Acc=0.3400  Prec=0.3803  Rec=0.0624  F1=0.0598
RandomForest  Acc=0.2973  Prec=0.0874  Rec=0.0375  F1=0.0197


Running all models (KNN, SVM, Naive Bayes and Random Forest) on the Testing phase

In [148]:
# Testing Phase
print("  Evaluation on TEST dataset (trained on TRAIN only)  ")
for name, model in models.items():
    if name == "KNN":
        Xtr, ytr, Xt = X_train_knn, y_train_knn, X_test
    elif name == "RandomForest":
        Xtr, ytr, Xt = X_train_rf, y_train_rf, X_test
    elif name == "LogReg_SVD":
        Xtr, ytr, Xt = X_train_svd, y_train, X_test_svd
    elif name == "LogReg_W2V":
        Xtr, ytr, Xt = X_train_w2v, y_train, X_test_w2v
    else:
        Xtr, ytr, Xt = X_train, y_train, X_test

    acc, pr, rc, f1 = evaluate_model(model, Xtr, ytr, Xt, y_test)
    print(f"{name:12s}  Acc={acc:.4f}  Prec={pr:.4f}  Rec={rc:.4f}  F1={f1:.4f}")

    # store for table
    results_test.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": pr,
        "Recall": rc,
        "F1": f1,
    })

  Evaluation on TEST dataset (trained on TRAIN only)  
KNN           Acc=0.3155  Prec=0.3061  Rec=0.1123  F1=0.1399
SVM           Acc=0.4920  Prec=0.4437  Rec=0.3441  F1=0.3729


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


LogReg_SVD    Acc=0.4807  Prec=0.3725  Rec=0.2510  F1=0.2723


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


LogReg_W2V    Acc=0.4056  Prec=0.3250  Rec=0.1727  F1=0.1982
NaiveBayes    Acc=0.3381  Prec=0.3377  Rec=0.0606  F1=0.0572
RandomForest  Acc=0.3022  Prec=0.0948  Rec=0.0385  F1=0.0217


Creating and Saving the Result Tables

In [149]:
# Table for tuning
dev_table = pd.DataFrame(results_dev).set_index("Model").sort_values("F1", ascending=False)

# Table for testing
test_table = pd.DataFrame(results_test).set_index("Model").sort_values("F1", ascending=False)

# Save to csv
os.makedirs("results", exist_ok=True)
dev_table.to_csv("results/goemotions_baselines_dev.csv")
test_table.to_csv("results/goemotions_baselines_test.csv")

print("Saved: results/goemotions_baselines_dev.csv")
print("Saved: results/goemotions_baselines_test.csv")

Saved: results/goemotions_baselines_dev.csv
Saved: results/goemotions_baselines_test.csv


Displaying the Result Tables

In [150]:
try:
    from tabulate import tabulate
    print("\nDEV Table:\n", tabulate(dev_table.reset_index(), headers="keys", tablefmt="github", floatfmt=".4f"))
    print("\nTEST Table:\n", tabulate(test_table.reset_index(), headers="keys", tablefmt="github", floatfmt=".4f"))
except Exception as e:
    print("Pretty print failed:", e)
    print("\nDEV Table:\n", dev_table)
    print("\nTEST Table:\n", test_table)


DEV Table:
 |    | Model        |   Accuracy |   Precision |   Recall |     F1 |
|----|--------------|------------|-------------|----------|--------|
|  0 | SVM          |     0.4884 |      0.4202 |   0.3194 | 0.3451 |
|  1 | LogReg_SVD   |     0.4921 |      0.4350 |   0.2693 | 0.3000 |
|  2 | LogReg_W2V   |     0.4062 |      0.3283 |   0.1793 | 0.2077 |
|  3 | KNN          |     0.3111 |      0.3257 |   0.1146 | 0.1451 |
|  4 | NaiveBayes   |     0.3400 |      0.3803 |   0.0624 | 0.0598 |
|  5 | RandomForest |     0.2973 |      0.0874 |   0.0375 | 0.0197 |

TEST Table:
 |    | Model        |   Accuracy |   Precision |   Recall |     F1 |
|----|--------------|------------|-------------|----------|--------|
|  0 | SVM          |     0.4920 |      0.4437 |   0.3441 | 0.3729 |
|  1 | LogReg_SVD   |     0.4807 |      0.3725 |   0.2510 | 0.2723 |
|  2 | LogReg_W2V   |     0.4056 |      0.3250 |   0.1727 | 0.1982 |
|  3 | KNN          |     0.3155 |      0.3061 |   0.1123 | 0.1399 |
|  4 | 

# roBERTa fine-tuning

Tokenizer + Tokenization

In [151]:
model_name = "roberta-large"
max_length = 128

tokenizer = AutoTokenizer.from_pretrained(model_name)

train_encodings = tokenizer(
    list(X_train_text),
    truncation=True,
    max_length=max_length
)

dev_encodings = tokenizer(
    list(X_dev_text),
    truncation=True,
    max_length=max_length
)

test_encodings = tokenizer(
    list(X_test_text),
    truncation=True,
    max_length=max_length
)

Convert to HuggingFace Dataset format (important for Trainer)

In [152]:
train_dataset = HFDataset.from_dict({
    "input_ids": train_encodings["input_ids"],
    "attention_mask": train_encodings["attention_mask"],
    "labels": y_train_ml
})

dev_dataset = HFDataset.from_dict({
    "input_ids": dev_encodings["input_ids"],
    "attention_mask": dev_encodings["attention_mask"],
    "labels": y_dev_ml
})

test_dataset = HFDataset.from_dict({
    "input_ids": test_encodings["input_ids"],
    "attention_mask": test_encodings["attention_mask"],
    "labels": y_test_ml
})

Dynamic padding

In [153]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Preparing the train dataset for fine-tuning

Load Model

In [154]:
num_labels = len(set(y_train))

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Metric Function

In [155]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))          # sigmoid
    preds = (probs >= 0.5).astype(int)         # threshold
    labels = labels.astype(int)

    return {
        "f1_micro": f1_score(labels, preds, average="micro", zero_division=0),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
    }

Training Arguments

In [156]:
training_args = TrainingArguments(
    output_dir="./bert_bsp_s5",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    num_train_epochs=5,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    weight_decay=0.01,
    warmup_ratio=0.1,

    logging_steps=50,
    fp16=torch.cuda.is_available(),
    seed=SEED
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


WeightedTrainer

In [157]:
class WeightedTrainer(Trainer):
    def __init__(self, *args, pos_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight  # can start on CPU; we'll move it at loss time

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels") if "labels" in inputs else inputs.pop("label")
        outputs = model(**inputs)
        logits = outputs.logits

        labels = labels.float()

        # 🔧 Ensure pos_weight is on same device as logits
        pos_weight = self.pos_weight.to(logits.device)

        loss_fct = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

Trainer Setup

In [158]:
pos_weight = torch.tensor(pos_weight_np).to(model.device)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    pos_weight=pos_weight
)


In [159]:
def get_dev_probs(trainer, dataset):
    preds = trainer.predict(dataset)

    # model outputs → probabilities
    probs = torch.sigmoid(
        torch.tensor(preds.predictions)
    ).numpy()

    # HuggingFace Dataset column → NumPy array
    y_true = np.array(dataset["labels"])

    return probs, y_true

In [160]:
def tune_thresholds(probs, y_true, labels, steps=50):
    thresholds = {}

    for i, label in enumerate(labels):
        best_f1 = 0.0
        best_t = 0.5

        for t in np.linspace(0.05, 0.95, steps):
            y_pred = (probs[:, i] >= t).astype(int)
            f1 = f1_score(y_true[:, i], y_pred, zero_division=0)

            if f1 > best_f1:
                best_f1 = f1
                best_t = t

        thresholds[label] = best_t

    return thresholds

In [161]:
dev_probs, dev_true = get_dev_probs(trainer, dev_dataset)
LABEL_THRESHOLDS = tune_thresholds(dev_probs, dev_true, LABELS)

Fine-tuning RoBERTa

In [162]:
print("Fine-tuning RoBERTa...")
trainer.train()

Fine-tuning RoBERTa...


Epoch,Training Loss,Validation Loss,Model Preparation Time,F1 Micro,F1 Macro
1,0.649232,0.626757,0.006400,0.391057,0.371554
2,0.636494,0.663918,0.006400,0.417251,0.407408
3,0.423902,0.604957,0.006400,0.430654,0.406633
4,0.341373,0.687702,0.006400,0.498958,0.451784
5,0.280738,0.747550,0.006400,0.504965,0.460400


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=13570, training_loss=0.5181994745286844, metrics={'train_runtime': 3334.6487, 'train_samples_per_second': 65.089, 'train_steps_per_second': 4.069, 'total_flos': 1.3300262062992432e+16, 'train_loss': 0.5181994745286844, 'epoch': 5.0})

Tune a global threshold

In [163]:
pred = trainer.predict(dev_dataset)
logits = pred.predictions
labels = pred.label_ids.astype(int)
probs = 1 / (1 + np.exp(-logits))

thresholds = np.zeros(probs.shape[1], dtype=np.float32)

for j in range(probs.shape[1]):
    best_t, best_f1 = 0.5, -1
    for t in np.arange(0.05, 0.95, 0.05):
        preds_j = (probs[:, j] >= t).astype(int)
        f1_j = f1_score(labels[:, j], preds_j, zero_division=0)
        if f1_j > best_f1:
            best_f1 = f1_j
            best_t = t
    thresholds[j] = best_t

Evaluate dev/test using the best threshold

In [164]:
def eval_with_label_thresholds(dataset, thresholds):
    pred = trainer.predict(dataset)
    logits = pred.predictions
    labels = pred.label_ids.astype(int)
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= thresholds).astype(int)
    return {
        "f1_micro": f1_score(labels, preds, average="micro", zero_division=0),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
    }

print("DEV:", eval_with_label_thresholds(dev_dataset, thresholds))
print("TEST:", eval_with_label_thresholds(test_dataset, thresholds))

DEV: {'f1_micro': 0.5984732824427481, 'f1_macro': 0.5402015857754022}


TEST: {'f1_micro': 0.597693531283139, 'f1_macro': 0.521115893075048}


Evaluate on DEV dataset

In [165]:
print("DEV evaluation:")
dev_metrics = trainer.evaluate(eval_dataset=dev_dataset)
dev_metrics

DEV evaluation:


{'eval_loss': 0.7475497126579285,
 'eval_model_preparation_time': 0.0064,
 'eval_f1_micro': 0.5049649679923384,
 'eval_f1_macro': 0.46040043988106,
 'eval_runtime': 11.3755,
 'eval_samples_per_second': 476.989,
 'eval_steps_per_second': 29.889,
 'epoch': 5.0}

Evaluate on TEST dataset

In [166]:
print("TEST evaluation:")
test_metrics = trainer.evaluate(eval_dataset=test_dataset)
test_metrics

TEST evaluation:


{'eval_loss': 0.7490428686141968,
 'eval_model_preparation_time': 0.0064,
 'eval_f1_micro': 0.5028588777007539,
 'eval_f1_macro': 0.4495422109330281,
 'eval_runtime': 11.9398,
 'eval_samples_per_second': 454.532,
 'eval_steps_per_second': 28.476,
 'epoch': 5.0}

# RAG-based Justification

Retrieval-Augmented Generation (RAG) for Emotion Justification

In [167]:
# Retriever encoder
retriever_model = SentenceTransformer("all-MiniLM-L6-v2")

# Retrieval corpus = raw training texts + their multi-label targets
texts = X_train_text          # list[str]
labels = y_train_ml           # multi-hot (n_samples, n_labels)

# sanity checks
print("texts:", type(texts), len(texts))
print("labels:", type(labels), labels.shape)

# Encode + build FAISS index (cosine similarity via normalized inner product)
embeddings = retriever_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)
embeddings = np.asarray(embeddings, dtype="float32")

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


texts: <class 'list'> 43410
labels: <class 'numpy.ndarray'> (43410, 28)


Batches:   0%|          | 0/1357 [00:00<?, ?it/s]

Retrieve similar emotionally-relevant posts

In [168]:
def retrieve_similar_posts_filtered(query_text, predicted_emotions, top_k=5, fetch_k=30):
    # Retrieve more than needed…
    q_emb = retriever_model.encode([query_text], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q_emb, fetch_k)

    pred_set = set(predicted_emotions)

    filtered = []
    for score, idx in zip(scores[0], ids[0]):
        ex_labels = [label_names[i] for i in np.where(labels[idx] == 1)[0]]
        overlap = len(pred_set.intersection(ex_labels))
        filtered.append({
            "score": float(score),
            "overlap": overlap,
            "text": texts[idx],
            "labels": ex_labels
        })

    # Rank: highest overlap first, then similarity
    filtered.sort(key=lambda x: (x["overlap"], x["score"]), reverse=True)

    return filtered[:top_k]

Evidence Keyword Extraction

In [169]:
STOP = set("""
a an and are as at be but by for if in into is it no not of on or such that the their then there these they this to was will with
i im i've id dont cant didnt wont isnt arent wasnt werent
""".split())

def extract_keywords(texts, top_k=12):
    tokens = []
    for t in texts:
        t = t.lower()
        t = re.sub(r"[^a-z\s']", " ", t)
        toks = [w for w in t.split() if len(w) >= 4 and w not in STOP]
        tokens.extend(toks)
    return [w for w, _ in Counter(tokens).most_common(top_k)]

Generate Justification

In [234]:
def generate_justification(post, predicted_emotions, retrieved_examples, k=2, max_chars=220):
    # preserve original indices from the retrieved list
    picked = list(enumerate(retrieved_examples, 1))[:k]

    lines = []
    for orig_i, ex in picked:
        snippet = ex["text"].replace("\n", " ").strip()
        snippet = snippet[:max_chars] + ("..." if len(snippet) > max_chars else "")
        lines.append(f"[Ex{orig_i}] {snippet}")

    return "\n".join(lines)

Retrieved Evidence

In [245]:
def show_retrieved(retrieved, n=5):
    print("\nRetrieved similar posts:")
    for i, ex in enumerate(retrieved[:n], 1):
        print(f"(Ex{i}) labels={ex['labels']} | {ex['text'][:120]}...")

# show_retrieved(retrieved) in the Final Pipeline

In [237]:
trainer.model.config.id2label = {i: LABELS[i] for i in range(NUM_LABELS)}
trainer.model.config.label2id = {LABELS[i]: i for i in range(NUM_LABELS)}

print("Label mapping fixed:")
print(trainer.model.config.id2label)

Label mapping fixed:
{0: 'admiration', 1: 'amusement', 2: 'anger', 3: 'annoyance', 4: 'approval', 5: 'caring', 6: 'confusion', 7: 'curiosity', 8: 'desire', 9: 'disappointment', 10: 'disapproval', 11: 'disgust', 12: 'embarassment', 13: 'excitement', 14: 'fear', 15: 'gratitude', 16: 'grief', 17: 'joy', 18: 'love', 19: 'nervousness', 20: 'optimism', 21: 'pride', 22: 'realization', 23: 'relief', 24: 'remorse', 25: 'sadness', 26: 'surprise', 27: 'neutral'}


RoBERTa inference helper

In [238]:
# Label names from the trained RoBERTa model
label_names = LABELS

print("Number of labels:", len(label_names))
print(label_names)

Number of labels: 28
['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']


switch generator to FLAN-T5 (instruction-following)
- Much cleaner, consistent justifications

In [239]:
def predict_emotions(text, top_k_fallback=2):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}

    with torch.no_grad():
        logits = trainer.model(**inputs).logits
        probs = torch.sigmoid(logits)[0].cpu().numpy()

    # normal per-label thresholding
    predicted = [
        (LABELS[i], float(probs[i]))
        for i in range(NUM_LABELS)
        if float(probs[i]) >= LABEL_THRESHOLDS[LABELS[i]]
    ]

    # fallback if nothing passes thresholds
    if len(predicted) == 0 and top_k_fallback is not None and top_k_fallback > 0:
        top_idx = np.argsort(-probs)[:top_k_fallback]
        predicted = [(LABELS[i], float(probs[i])) for i in top_idx]

    return {"predicted": predicted, "all_probs": dict(zip(LABELS, probs))}

Final Pipeline for RAG-based Justification (Example)

In [248]:
text = "I feel empty and exhausted, nothing makes sense anymore."

predicted = predict_emotions(text)["predicted"]
pred_labels = [e for e, _ in predicted]

retrieved = retrieve_similar_posts_filtered(text, pred_labels, top_k=5, fetch_k=30)

justification = generate_justification(text, pred_labels, retrieved, k=2)

print("Predicted emotions:", pred_labels)
show_retrieved(retrieved)
print("\nJustification:\n", justification)


Predicted emotions: ['annoyance', 'disappointment', 'nervousness', 'sadness', 'neutral']

Retrieved similar posts
(Ex1) labels=['annoyance', 'disappointment', 'sadness'] | Virtually no happiness anymore, exhausted with trying, meds not working properly and just tired of it all....
(Ex2) labels=['disappointment'] | Living is exhausting....
(Ex3) labels=['neutral'] | I've isolated myself to the extent that i only focus my energy on myself/work.as lonely as it sounds its been such a bre...
(Ex4) labels=['neutral'] | It's like how I feel after I eat McDonalds. Satisfied, disappointed, sad, full, tired....
(Ex5) labels=['sadness'] | I'm kind of struggling tonight. Bored, lonely... man...

Justification:
 [Ex1] Virtually no happiness anymore, exhausted with trying, meds not working properly and just tired of it all.
[Ex2] Living is exhausting.
